In [1]:
import json
import tkinter as tk
from tkinter import ttk, messagebox
import socket

PORT = 5050
SERVER = socket.gethostbyname(socket.gethostname())
ADDR = (SERVER, PORT)
HEADER = 64
FORMAT = 'utf-8'

class NewLineCinemaApp:
    def __init__(self, master):
        self.master = master
        master.title("NewLine Cinema Ticket System")
        master.geometry("800x600")

        master.columnconfigure(0, weight=1)
        master.rowconfigure(0, weight=1)
        master.rowconfigure(1, weight=2)

        self.sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self.sock.connect(ADDR)

        self.movies = []

        purchase_frame = ttk.LabelFrame(master, text="Purchase Tickets", padding=10)
        purchase_frame.grid(row=0, column=0, padx=10, pady=10, sticky="nsew")
        purchase_frame.columnconfigure(1, weight=1)

        ttk.Label(purchase_frame, text="Select Movie:").grid(row=0, column=0, sticky="e", padx=5, pady=5)
        self.movie_dropdown = ttk.Combobox(purchase_frame, state="readonly")
        self.movie_dropdown.grid(row=0, column=1, sticky="ew", padx=5, pady=5)

        ttk.Label(purchase_frame, text="Customer Name:").grid(row=1, column=0, sticky="e", padx=5, pady=5)
        self.customer_entry = ttk.Entry(purchase_frame)
        self.customer_entry.grid(row=1, column=1, sticky="ew", padx=5, pady=5)

        ttk.Label(purchase_frame, text="Tickets:").grid(row=2, column=0, sticky="e", padx=5, pady=5)
        self.tickets_entry = ttk.Entry(purchase_frame)
        self.tickets_entry.grid(row=2, column=1, sticky="ew", padx=5, pady=5)

        self.purchase_button = ttk.Button(purchase_frame, text="Purchase Tickets", command=self.purchase_tickets)
        self.purchase_button.grid(row=3, column=0, columnspan=2, pady=10, sticky="ew")

        admin_frame = ttk.LabelFrame(master, text="Manage Movies", padding=10)
        admin_frame.grid(row=1, column=0, padx=10, pady=10, sticky="nsew")
        admin_frame.columnconfigure(1, weight=1)
        admin_frame.columnconfigure(2, weight=0)

        labels = ["ID", "Title", "Room", "Release", "End", "Tickets", "Price"]
        self.entries = {}

        for i, label in enumerate(labels):
            ttk.Label(admin_frame, text=label + ":").grid(row=i, column=0, sticky="e", padx=5, pady=2)
            entry = ttk.Entry(admin_frame)
            entry.grid(row=i, column=1, sticky="ew", padx=5, pady=2)
            self.entries[label.lower()] = entry

        ttk.Button(admin_frame, text="Add", command=self.add_movie).grid(row=0, column=2, padx=5, pady=2, sticky="ew")
        ttk.Button(admin_frame, text="Update", command=self.update_movie).grid(row=1, column=2, padx=5, pady=2, sticky="ew")
        ttk.Button(admin_frame, text="Delete", command=self.delete_movie).grid(row=2, column=2, padx=5, pady=2, sticky="ew")

        self.refresh_movies()

    def send_request(self, action, data=None):
        try:
            msg = json.dumps({"action": action, "data": data})
            msg_length = len(msg)
            send_header = str(msg_length).encode(FORMAT)
            send_header += b' ' * (HEADER - len(send_header))
            self.sock.send(send_header)
            self.sock.send(msg.encode(FORMAT))
            response_length = self.sock.recv(HEADER).decode(FORMAT)
            if response_length:
                response = self.sock.recv(int(response_length)).decode(FORMAT)
                return json.loads(response)
        except Exception as e:
            return {"status": "error", "message": str(e)}

    def fetch_movies(self):
        response = self.send_request("Retrieve The List of Movies")
        if response["status"].lower().startswith("success"):
            self.movies = response["Current Movies Showing"]
            self.movie_dropdown.set('')
            self.movie_dropdown["values"] = [movie[1] for movie in self.movies]
        else:
            messagebox.showerror("Error", response.get("message", "Failed to retrieve movies."))

    def refresh_movies(self):
        self.fetch_movies()

    def purchase_tickets(self):
        try:
            index = self.movie_dropdown.current()
            if index == -1:
                raise ValueError("Please select a movie.")

            movie_id = self.movies[index][0]
            customer = self.customer_entry.get()
            tickets = int(self.tickets_entry.get())

            if not customer or tickets <= 0:
                raise ValueError("Enter a valid customer name and number of tickets.")

            response = self.send_request("Record a Ticket Sale and Number of Tickets", {
                "movie_id": movie_id,
                "customer_name": customer,
                "number_of_tickets": tickets
            })

            if response.get("Order Status", "").lower().startswith("successful"):
                messagebox.showinfo("Success", f"Total: R{response['Total']:.2f}\nSaved as: {response['Movie Ticket']}")
                self.refresh_movies()

                
                try:
                    with open(response["Movie Ticket"], "r") as file:     #This will be used to display the order summary/reciept for the client in a seperate pop-up window 
                        ticket_content = file.read()

                    ticket_window = tk.Toplevel(self.master)
                    ticket_window.title("Your Movie Ticket")
                    ticket_window.geometry("400x300")
                    ticket_window.transient(self.master)

                    text_widget = tk.Text(ticket_window, wrap=tk.WORD)
                    text_widget.insert(tk.END, ticket_content)
                    text_widget.config(state=tk.DISABLED)
                    text_widget.pack(expand=True, fill=tk.BOTH, padx=10, pady=10)

                except Exception as e:
                    messagebox.showerror("Error", f"Could not open ticket file: {e}")
            else:
                messagebox.showerror("Error", response.get("message", "Purchase failed."))

        except ValueError as ve:
            messagebox.showerror("Input Error", str(ve))

    def add_movie(self):
        try:
            data = self.get_movie_entry_data(include_id=False)
            response = self.send_request("Add A New Movie", data)
            if response["status"].lower().startswith("success"):
                messagebox.showinfo("Success", "Movie added successfully.")
                self.refresh_movies()
            else:
                messagebox.showerror("Error", response.get("message", "Failed to add movie."))
        except ValueError as ve:
            messagebox.showerror("Input Error", str(ve))

    def update_movie(self):
        try:
            data = self.get_movie_entry_data(include_id=True)
            response = self.send_request("Update Movie Details", data)
            if response["status"].lower().startswith("success"):
                messagebox.showinfo("Success", "Movie updated successfully.")
                self.refresh_movies()
            else:
                messagebox.showerror("Error", response.get("message", "Failed to update movie."))
        except ValueError as ve:
            messagebox.showerror("Input Error", str(ve))

    def delete_movie(self):
        try:
            movie_id = int(self.entries["id"].get())
            response = self.send_request("Delete A Movie", {"id": movie_id})
            if response["status"].lower().startswith("success"):
                messagebox.showinfo("Success", "Movie deleted successfully.")
                self.refresh_movies()
            else:
                messagebox.showerror("Error", response.get("message", "Failed to delete movie."))
        except ValueError:
            messagebox.showerror("Input Error", "Please enter a valid movie ID.")

    def get_movie_entry_data(self, include_id):
        try:
            data = {
                "title": self.entries["title"].get(),
                "cinema_room": int(self.entries["room"].get()),
                "release_date": self.entries["release"].get(),
                "end_date": self.entries["end"].get(),
                "tickets_available": int(self.entries["tickets"].get()),
                "ticket_price": float(self.entries["price"].get()),
            }
            if include_id:
                data["id"] = int(self.entries["id"].get())
            return data
        except Exception:
            raise ValueError("Please fill in all fields correctly.")

def main():
    root = tk.Tk()
    app = NewLineCinemaApp(root)
    root.mainloop()

if __name__ == "__main__":
    main()
